In [1]:
print("hello")

hello


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../data/raw/crashes_raw.csv", low_memory=False)
print(f"Raw: {len(df):,} rows")

# parse dates
df['crash_date'] = pd.to_datetime(df['crash_date'])
df['crash_datetime'] = pd.to_datetime(
    df['crash_date'].dt.strftime('%Y-%m-%d') + ' ' + df['crash_time'],
    errors='coerce'
)

# drop rows with no GPS
before = len(df)
df = df[df['latitude'].notnull() & df['longitude'].notnull()
        & (df['latitude'] != 0) & (df['longitude'] != 0)]
print(f"Dropped {before - len(df):,} rows without coordinates. Remaining: {len(df):,}")

# drop columns we don't need
drop_cols = [
    'location',
    'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5',
    'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4',
    'contributing_factor_vehicle_5',
    'off_street_name'
]
df = df.drop(columns=drop_cols)

# fix one null in injuries
df['number_of_persons_injured'] = df['number_of_persons_injured'].fillna(0).astype(int)

# clean borough text
df['borough'] = df['borough'].str.strip().str.title()

# Collision Severity Index
# every crash = 1 point, each injury = 2, each fatality = 5
df['csi'] = 1 + 2 * df['number_of_persons_injured'] + 5 * df['number_of_persons_killed']

print("\nCSI distribution:")
print(df['csi'].describe().round(2))
print(f"Crashes with injuries/fatalities: {(df['csi'] > 1).sum():,}")

# save
df.to_csv("../data/processed/crashes_cleaned.csv", index=False)
print(f"\nSaved to data/processed/crashes_cleaned.csv")
print(f"Final shape: {df.shape}")

Raw: 500,000 rows
Dropped 38,575 rows without coordinates. Remaining: 461,425

CSI distribution:
count    461425.00
mean          2.08
std           1.68
min           1.00
25%           1.00
50%           1.00
75%           3.00
max          69.00
Name: csi, dtype: float64
Crashes with injuries/fatalities: 186,207

Saved to data/processed/crashes_cleaned.csv
Final shape: (461425, 23)
